# Phase 4: Anomaly Detection (Second-Opinion Flag)

The supervised models say *"I expected X, you charged Y"*. An unsupervised detector says *"this shipment is unusual."* Their intersection is the high-precision audit queue.

Two detectors:
- **Isolation Forest** (`sklearn.ensemble.IsolationForest`, contamination=0.05) on the raw 105-feature parquet — tree-based, no scaling needed.
- **Autoencoder** (PyTorch Lightning, 105 → 64 → 16 → 64 → 105, MSE, AdamW + cosine LR, 30 epochs) on StandardScaler-normalised features. Reconstruction error per row is the anomaly score.

Both produce per-row scores. Each is **percentile-ranked against the train set**, then averaged to give a final `anomaly_score ∈ [0, 1]`. The threshold is calibrated so the combined `review_recommended` flag in `predict.py` fires on **2–8% of test shipments** (handoff success criterion).

Artifacts written:
```
models/isolation_forest.pkl
models/autoencoder.pt              # state_dict + hparams
models/scaler_v2.pkl               # StandardScaler fit on the v2 train.parquet
models/anomaly_threshold.json      # threshold + train-time score quantiles for percentile-rank lookup
```

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
MODELS = ROOT / 'models'

sys.path.insert(0, str(ROOT / 'src'))
from anomaly import AnomalyAutoencoder  # noqa: E402

train_df = pd.read_parquet(DATA / 'train.parquet')
val_df   = pd.read_parquet(DATA / 'val.parquet')
test_df  = pd.read_parquet(DATA / 'test.parquet')

TARGET_COLS = ['dim_flag', 'log_net_charge', 'Net Charge Billed Currency',
               'log_base_charge', 'log_misc_charge']
feature_cols = [c for c in train_df.columns if c not in TARGET_COLS]

X_train = train_df[feature_cols]
X_val   = val_df[feature_cols]
X_test  = test_df[feature_cols]

print(f'train {X_train.shape}  val {X_val.shape}  test {X_test.shape}')
print(f'features = {len(feature_cols)}')

train (45508, 105)  val (5688, 105)  test (5689, 105)
features = 105


## Isolation Forest

Raw-feature detector. `contamination=0.05` means the model assumes ~5% of the training data is anomalous when choosing its decision boundary — this isn't the final threshold, just a hint for the isolation depth normalisation.

In [2]:
iso = IsolationForest(
    n_estimators=200, contamination=0.05,
    random_state=42, n_jobs=-1,
)
iso.fit(X_train.values)

# More-anomalous rows get *lower* score_samples; flip sign so higher = more anomalous
iso_train = -iso.score_samples(X_train.values)
iso_test  = -iso.score_samples(X_test.values)

print(f'IF train score range: [{iso_train.min():.4f}, {iso_train.max():.4f}]')
print(f'IF test  top-5 quantiles: {np.quantile(iso_test, [0.5, 0.75, 0.9, 0.95, 0.99]).round(4)}')

IF train score range: [0.3392, 0.5581]
IF test  top-5 quantiles: [0.38   0.3943 0.4103 0.4213 0.4644]


## StandardScaler for the autoencoder

Phase 1 saved scaled parquets but they were derived from a scaler we never persisted under the v2 schema. Refit one here on the unscaled train parquet and save it for inference.

In [3]:
scaler = StandardScaler()
scaler.fit(X_train.values)

X_train_s = scaler.transform(X_train.values).astype(np.float32)
X_val_s   = scaler.transform(X_val.values).astype(np.float32)
X_test_s  = scaler.transform(X_test.values).astype(np.float32)

joblib.dump(scaler, MODELS / 'scaler_v2.pkl')
print(f'scaler saved · feature means range [{scaler.mean_.min():.2f}, {scaler.mean_.max():.2f}]')

scaler saved · feature means range [-112.91, 13994.73]


## Autoencoder

Architecture: `105 → 64 → 16 → 64 → 105`, ReLU between layers, MSE reconstruction loss, AdamW + cosine LR over 30 epochs. The `AnomalyAutoencoder` class lives in `src/anomaly.py` so the same definition is used at inference time.

In [4]:
pl.seed_everything(42, workers=True)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_s)),
    batch_size=512, shuffle=True, num_workers=0,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val_s)),
    batch_size=512, shuffle=False, num_workers=0,
)

ae = AnomalyAutoencoder(
    input_dim=X_train_s.shape[1], hidden_dim=64, latent_dim=16,
    lr=1e-3, max_epochs=30,
)

trainer = pl.Trainer(
    max_epochs=30, accelerator='auto', devices=1,
    logger=False, enable_checkpointing=False,
    enable_progress_bar=True, log_every_n_steps=20,
)
trainer.fit(ae, train_loader, val_loader)

# Persist state_dict + hparams (decouples checkpoint from Lightning trainer state)
torch.save(
    {'state_dict': ae.state_dict(), 'hparams': dict(ae.hparams)},
    MODELS / 'autoencoder.pt',
)
print('autoencoder saved')

Seed set to 42


GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ Sequential │  7.8 K │ train │     0 │
│ 1 │ decoder │ Sequential │  7.9 K │ train │     0 │
│ 2 │ loss_fn │ MSELoss    │      0 │ train │     0 │
└───┴─────────┴────────────┴────────┴───────┴───────┘

Trainable params: 15.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 15.7 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/anaconda3/envs/fedex-ml/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/opt/anaconda3/envs/fedex-ml/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:43
4: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/opt/anaconda3/envs/fedex-ml/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:43
4: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

In [5]:
ae_train = ae.reconstruction_error(torch.from_numpy(X_train_s))
ae_test  = ae.reconstruction_error(torch.from_numpy(X_test_s))
print(f'AE train MSE range: [{ae_train.min():.4f}, {ae_train.max():.4f}]')
print(f'AE test  top-5 quantiles: {np.quantile(ae_test, [0.5, 0.75, 0.9, 0.95, 0.99]).round(4)}')

AE train MSE range: [0.0087, 430.7999]
AE test  top-5 quantiles: [0.1033 0.1732 0.2745 0.4763 1.4307]


## Combine + calibrate threshold

Percentile-rank both scores against the *train* distribution, average them, then pick the threshold so the test-set fire-rate sits in the 2–8% band. We start at the 95th percentile (the natural "top 5%" target) and only adjust if we fall outside [0.02, 0.08].

In [6]:
iso_ref = np.sort(iso_train)
ae_ref  = np.sort(ae_train)

def pct_rank(reference, values):
    return np.searchsorted(reference, values, side='right') / len(reference)

iso_pct_test = pct_rank(iso_ref, iso_test)
ae_pct_test  = pct_rank(ae_ref,  ae_test)
combined_test = (iso_pct_test + ae_pct_test) / 2

threshold = 0.95
fire_rate = float((combined_test >= threshold).mean())
print(f'initial threshold=0.95  →  test fire rate = {fire_rate:.4f}')

# Walk the threshold up or down until we land in [0.02, 0.08]
if fire_rate < 0.02:
    while fire_rate < 0.02 and threshold > 0.80:
        threshold -= 0.01
        fire_rate = float((combined_test >= threshold).mean())
elif fire_rate > 0.08:
    while fire_rate > 0.08 and threshold < 0.999:
        threshold += 0.005
        fire_rate = float((combined_test >= threshold).mean())

print(f'tuned threshold={threshold:.3f}  →  test fire rate = {fire_rate:.4f}')
assert 0.02 <= fire_rate <= 0.08, f'fire rate {fire_rate} outside [0.02, 0.08] band'

initial threshold=0.95  →  test fire rate = 0.0272
tuned threshold=0.950  →  test fire rate = 0.0272


In [7]:
joblib.dump(iso, MODELS / 'isolation_forest.pkl')

with open(MODELS / 'anomaly_threshold.json', 'w') as f:
    json.dump({
        'threshold': threshold,
        'test_fire_rate': fire_rate,
        'iso_reference_scores': iso_ref.tolist(),
        'ae_reference_scores':  ae_ref.tolist(),
    }, f)

print('Phase 4 artifacts written:')
for p in ['isolation_forest.pkl', 'autoencoder.pt', 'scaler_v2.pkl',
          'anomaly_threshold.json']:
    size_kb = (MODELS / p).stat().st_size / 1024
    print(f'  {p:30s} {size_kb:>8.1f} KB')

Phase 4 artifacts written:
  isolation_forest.pkl             1648.1 KB
  autoencoder.pt                     65.3 KB
  scaler_v2.pkl                       3.0 KB
  anomaly_threshold.json           1832.6 KB


## Smoke test the `src.anomaly.score_shipment` contract

Pull a single row from `test.parquet`, score it through the public API, and confirm the output schema.

In [8]:
# Force a fresh import so the lru_cache picks up the artifacts we just wrote
import importlib, sys
if 'anomaly' in sys.modules:
    importlib.reload(sys.modules['anomaly'])
import anomaly  # noqa: E402

sample_X = X_test.iloc[[0]].copy()
result = anomaly.score_shipment(sample_X)
print(json.dumps(result, indent=2, default=float))

{
  "anomaly_score": 0.9361211215610442,
  "anomaly_flagged": false,
  "iso_score": 0.963940406082447,
  "ae_score": 0.9083018370396414,
  "threshold": 0.95
}
